# OrionPulse Data Agent — Demo Notebook

This notebook walks through the full agent lifecycle end-to-end:

1. Seed the database
2. Deterministic KPI summary
3. Forecasting with confidence bands
4. Anomaly detection
5. Period-over-period comparison (new `compare` intent)
6. LLM orchestration mode (requires `ORION_LLM_API_KEY` or Ollama)
7. Planner trace — see every step the agent takes
8. Conversation memory
9. Dashboard spec generation

**Prerequisites:** `pip install -r requirements.txt` and `python data/init_db.py`

In [ ]:
import sys
from pathlib import Path

# Add project root to path so src/ imports work from the notebook
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

## 1. Seed the database

Creates the star schema (`dim_product`, `dim_region`, `fact_sales`), applies the three analytical views, and seeds synthetic sales data.

In [ ]:
import subprocess
result = subprocess.run(['python', str(ROOT / 'data' / 'init_db.py')], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 2. Deterministic KPI Summary

No LLM required. Runs SQL against the SQLite star schema and returns monthly KPIs.

In [ ]:
from src.orion_sales_agent.config import settings
from src.orion_sales_agent.analytics import kpi_summary
import pandas as pd

rows = kpi_summary(settings.db_path, grain='month')
df = pd.DataFrame(rows)
print(f'{len(df)} months of data')
df.tail(6)

In [ ]:
# Plot the revenue trend inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df['period'], df['net_revenue'], marker='o', linewidth=2)
ax.set_title('Monthly Net Revenue')
ax.set_xlabel('Period')
ax.set_ylabel('Net Revenue ($)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 3. Forecasting with Confidence Bands

Holt-Winters ETS with holdout backtest and automatic model selection between Holt-Linear and Holt-Winters based on RMSE.

In [ ]:
from src.orion_sales_agent.forecasting import forecast_metric

fc = forecast_metric(settings.db_path, metric='net_revenue', horizon=6)
diag = fc['diagnostics']
print(f"Method selected:  {diag['method']}")
print(f"Backtest MAPE:    {diag['mape']:.1f}%" if diag['mape'] else 'MAPE: n/a')
print(f"Backtest RMSE:    {diag['rmse']:.0f}" if diag['rmse'] else 'RMSE: n/a')
print(f"Candidates tried: {[c['method'] for c in diag['candidates']]}")

In [ ]:
# Plot history + forecast with 95% confidence band
hist = pd.DataFrame(fc['history'])
pred = pd.DataFrame(fc['forecast'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hist['period'], hist['value'], marker='o', label='History', linewidth=2)
ax.plot(pred['period'], pred['value'], marker='s', linestyle='--', label='Forecast', linewidth=2, color='orange')
ax.fill_between(pred['period'], pred['lower'], pred['upper'], alpha=0.25, color='orange', label='95% CI')
ax.set_title('Net Revenue Forecast (6 months)')
ax.set_xlabel('Period')
ax.set_ylabel('Net Revenue ($)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Anomaly Detection

Z-score thresholding over monthly aggregated metric.

In [ ]:
from src.orion_sales_agent.analytics import anomaly_detection

anomalies = anomaly_detection(settings.db_path, metric='net_revenue', threshold=2.0)
if anomalies:
    print(f'{len(anomalies)} anomalous period(s) detected:')
    for a in anomalies:
        direction = 'spike' if a['zscore'] > 0 else 'dip'
        print(f"  {a['period']}: ${a['value']:,.0f}  z={a['zscore']:.2f} ({direction})")
else:
    print('No anomalies detected at z=2.0 threshold.')

## 5. Period-over-Period Comparison (new intent)

The `compare` intent is a new deterministic handler that extracts period tokens from the question, runs `kpi_summary` for each, and diffs them. It also flags the note that the LLM path handles this more dynamically.

In [ ]:
from src.orion_sales_agent.agent import OrionAgent

agent = OrionAgent()

# Classify the intent first
question = 'compare 2024 vs 2025 revenue and margin'
intent = agent.classify_intent(question)
print(f'Intent classified as: {intent}')  # should be 'compare'

In [ ]:
# Run in deterministic mode
resp = agent.answer(question, mode='deterministic')
print('=== Answer ===')
print(resp.answer)
print()
print('=== Periods compared ===')
for period, rows in resp.data.get('periods', {}).items():
    total_rev = sum(r.get('net_revenue', 0) for r in rows)
    print(f'  {period}: {len(rows)} period(s), total revenue ${total_rev:,.0f}')

## 6. LLM Orchestration Mode

Requires either:
- `ORION_LLM_API_KEY` set in `.env` (OpenAI or compatible)
- Or Ollama running locally: `ORION_LLM_BASE_URL=http://localhost:11434/v1` with `ORION_LLM_MODEL=llama3.2`

The agent will gracefully fall back to deterministic mode if the LLM is unavailable.

In [ ]:
from src.orion_sales_agent.llm_client import llm_enabled

if llm_enabled():
    print('LLM is configured — will use LLM orchestration')
else:
    print('LLM not configured — will fall back to deterministic')
    print('To enable: set ORION_LLM_API_KEY in .env (or use Ollama — see .env.example)')

In [ ]:
# Ask a multi-step question in auto mode
# If LLM is configured: planner -> tools -> critic -> synthesizer
# If not:               deterministic fallback with fallback_reason set
resp_auto = agent.answer(
    'What drove the margin change between Q1 and Q2, and should I be concerned?',
    mode='auto',
)

print(f'Execution mode: {resp_auto.execution_mode}')
if resp_auto.fallback_reason:
    print(f'Fallback reason: {resp_auto.fallback_reason}')
print()
print('=== Answer ===')
print(resp_auto.answer)
print()
print('=== Reasoning summary ===')
for step in resp_auto.reasoning_summary:
    print(f'  • {step}')

## 7. Planner Trace (LLM mode)

Pass a `step_callback` to `answer()` to see each planner/tool/critic step as it happens. This is what `--trace` does in the CLI.

Only active when the LLM path runs. No-op in deterministic mode.

In [ ]:
trace_steps = []

def capture_step(msg: str) -> None:
    trace_steps.append(msg)
    print(msg)  # live output during execution

resp_traced = agent.answer(
    'forecast net revenue for next 3 months and flag any anomalies',
    mode='auto',
    step_callback=capture_step,
)

print()
print(f'Total trace steps captured: {len(trace_steps)}')
print(f'Execution mode: {resp_traced.execution_mode}')

## 8. Conversation Memory

The agent persists a bounded window (20 items, 50 KB) of Q&A to `data/agent_memory.json`. This demo shows the memory accumulating and then being cleared.

In [ ]:
import json
from src.orion_sales_agent.memory_store import load_memory
from src.orion_sales_agent.agent import MEMORY_FILE

mem = load_memory(MEMORY_FILE)
print(f'Memory has {len(mem)} item(s):')
for i, item in enumerate(mem[-3:], 1):
    print(f'  [{i}] Q: {item["question"][:60]}...')
    print(f'       intent={item["intent"]}')

In [ ]:
# Reset memory (same as: python scripts/ask_agent.py --reset-memory)
MEMORY_FILE.write_text('[]', encoding='utf-8')
print('Memory cleared.')
print(f'Items remaining: {len(load_memory(MEMORY_FILE))}')

## 9. Dashboard & Storyboard Spec Generation

In [ ]:
from src.orion_sales_agent.specs import dashboard_spec, storyboard_spec
import json

dash = dashboard_spec()
print(f'Dashboard spec: {len(dash["widgets"])} widgets')
for w in dash['widgets']:
    print(f"  [{w['type']}] {w['title']}")

In [ ]:
story = storyboard_spec(goal='Q2 business review for CFO', audience='exec', period='2025-Q2')
print(f'Storyboard: {len(story["slides"])} slides for goal: "{story["goal"]}"')
for s in story['slides']:
    print(f"  Slide {s['order']}: {s['title']}")

## Summary

| Feature | Module | LLM needed? |
|---------|--------|-------------|
| KPI summary | `analytics.py` | No |
| Forecasting (Holt-Winters + backtest) | `forecasting.py` | No |
| Anomaly detection | `analytics.py` | No |
| Period comparison | `agent.py` (`compare` intent) | No |
| Multi-step reasoning | `agent.py` (LLM path) | Yes |
| Planner trace | `agent.answer(step_callback=...)` | Yes (no-op otherwise) |
| Dashboard/storyboard specs | `specs.py` | No |
| Chart generation | `visualization.py` | No |

**The key design principle:** every question has a deterministic fallback. The LLM path adds multi-step reasoning and natural language synthesis but is never load-bearing.